In [ ]:
!pip install pillow tqdm nltk rouge bert_score
!pip install ipywidgets --upgrade
!pip install git+https://github.com/salaniz/pycocoevalcap.git
!jupyter nbextension enable --py widgetsnbextension --sys-prefix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import string
from PIL import Image
from pickle import dump, load

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import Input, Dense, Dropout, Embedding, LayerNormalization, Layer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import img_to_array, ImageDataGenerator
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.metrics import top_k_categorical_accuracy

from rouge import Rouge
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

from tqdm import tqdm
tqdm().pandas()

In [ ]:
def load_doc(filename):
    with open(filename, 'r') as file:
        text = file.read()
    return text

def all_img_captions(filename):
    file = load_doc(filename)
    captions = file.split('\n')
    descriptions = {}
    for caption in captions[:-1]:
        img, caption = caption.split('\t')
        if img[:-2] not in descriptions:
            descriptions[img[:-2]] = [caption]
        else:
            descriptions[img[:-2]].append(caption)
    return descriptions

def cleaning_text(captions):
    table = str.maketrans('', '', string.punctuation)
    for img, caps in captions.items():
        for i, img_caption in enumerate(caps):
            img_caption = img_caption.replace("-", " ")
            desc = img_caption.split()
            desc = [word.lower() for word in desc]
            desc = [word.translate(table) for word in desc]
            desc = [word for word in desc if len(word) > 1 and word.isalpha()]
            img_caption = ' '.join(desc)
            captions[img][i] = img_caption
    return captions

def text_vocabulary(descriptions):
    vocab = set()
    for key in descriptions.keys():
        [vocab.update(d.split()) for d in descriptions[key]]
    return vocab

def save_descriptions(descriptions, filename):
    lines = list()
    for key, desc_list in descriptions.items():
        for desc in desc_list:
            lines.append(key + '\t' + desc)
    data = "\n".join(lines)
    with open(filename, "w") as file:
        file.write(data)

In [ ]:
dataset_text = ""
dataset_images = "Flicker8k_Dataset"

filename = os.path.join(dataset_text, "Flickr8k.token.txt")
descriptions = all_img_captions(filename)
print("Length of descriptions :", len(descriptions))

In [ ]:
clean_descriptions = cleaning_text(descriptions)

vocabulary = text_vocabulary(clean_descriptions)
print("Length of vocabulary :", len(vocabulary))

save_descriptions(clean_descriptions, "descriptions.txt")

In [ ]:
MODEL = Xception(include_top=False, pooling='avg')

def extract_features(directory, augment=True, num_augmented=3):
    features = {}

    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    for img in tqdm(os.listdir(directory)):
        filename = os.path.join(directory, img)
        try:
            image = Image.open(filename).convert("RGB")
            image = image.resize((299, 299))
            image_array = img_to_array(image) / 127.5 - 1.0
            image_expanded = np.expand_dims(image_array, axis=0)

            feature = MODEL.predict(image_expanded, verbose=0)
            features[img] = feature

            if augment:
                i = 0
                for batch in datagen.flow(image_expanded, batch_size=1):
                    aug_feature = MODEL.predict(batch, verbose=0)
                    features[f"{img}_aug{i}"] = aug_feature
                    i += 1
                    if i >= num_augmented:
                        break
        except Exception as e:
            print(f"Error processing {img}: {e}")

    return features

features = extract_features(dataset_images, augment=True, num_augmented=3)

dump(features, open("features_augmented.p", "wb"))
features = load(open("features_augmented.p", "rb"))

In [ ]:
def load_photos(filename):
    file = load_doc(filename)
    photos = file.split("\n")[:-1]
    return photos

def load_clean_descriptions(filename, photos):
    file = load_doc(filename)
    descriptions = {}
    for line in file.split("\n"):
        words = line.split()
        if len(words) < 1:
            continue
        image, image_caption = words[0], words[1:]
        if image in photos:
            if image not in descriptions:
                descriptions[image] = []
            desc = '<start> ' + " ".join(image_caption) + ' <end>'
            descriptions[image].append(desc)
    return descriptions

def load_features(photos):
    all_features = load(open("features.p", "rb"))
    features = {k: all_features[k] for k in photos}
    return features

filename = os.path.join(dataset_text, "Flickr_8k.trainImages.txt")
train_imgs = load_photos(filename)
train_descriptions = load_clean_descriptions("descriptions.txt", train_imgs)
train_features = load_features(train_imgs)

In [ ]:
def dict_to_list(descriptions):
    all_desc = []
    for key in descriptions.keys():
        [all_desc.append(d) for d in descriptions[key]]
    return all_desc

def create_tokenizer(descriptions):
    desc_list = dict_to_list(descriptions)
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(desc_list)
    return tokenizer

tokenizer = create_tokenizer(train_descriptions)
dump(tokenizer, open('tokenizer.p', 'wb'))

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary Size :", vocab_size)

In [ ]:
def max_length_desc(descriptions):
    desc_list = dict_to_list(descriptions)
    return max(len(d.split()) for d in desc_list)

max_length = max_length_desc(descriptions)
print("Maximum Caption Length :", max_length)

In [ ]:
def create_sequences(tokenizer, max_length, desc_list, feature):
    X1, X2, y = list(), list(), list()
    for desc in desc_list:
        seq = tokenizer.texts_to_sequences([desc])[0]
        for i in range(1, len(seq)):
            in_seq, out_seq = seq[:i], seq[i]
            in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
            out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
            X1.append(feature)
            X2.append(in_seq)
            y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)

def data_generator(descriptions, features, tokenizer, max_length, batch_size=32):
    keys = list(descriptions.keys())
    while True:
        X1_batch, X2_batch, y_batch = [], [], []
        for key in keys:
            feature = features[key][0]
            X1, X2, y = create_sequences(tokenizer, max_length, descriptions[key], feature)
            if X1.shape[0] == 0:
                continue
            for i in range(X1.shape[0]):
                X1_batch.append(X1[i])
                X2_batch.append(X2[i])
                y_batch.append(y[i])
                if len(X1_batch) == batch_size:
                    yield ((np.array(X1_batch), np.array(X2_batch)), np.array(y_batch))
                    X1_batch, X2_batch, y_batch = [], [], []
        if len(X1_batch) > 0:
            yield ((np.array(X1_batch), np.array(X2_batch)), np.array(y_batch))

[a, b], c = next(data_generator(train_descriptions, features, tokenizer, max_length))
print("Shapes - Image Features :", a.shape, ", Sequences :", b.shape, ", Output :", c.shape)

In [ ]:
print('Dataset :', len(train_imgs))
print('Descriptions (train) :', len(train_descriptions))
print('Photos (train) :', len(train_features))
print('Vocabulary Size :', vocab_size)
print('Description Length : ', max_length)

In [ ]:
def masked_loss(y_true, y_pred):
    y_true_sparse = tf.argmax(y_true, axis=-1)
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true_sparse, y_pred, from_logits=False
    )
    mask = tf.cast(tf.not_equal(y_true_sparse, 0), dtype=loss.dtype)
    masked_loss = loss * mask
    return tf.reduce_mean(tf.boolean_mask(masked_loss, mask > 0))

def top_5_accuracy(y_true, y_pred):
    return top_k_categorical_accuracy(y_true, y_pred, k=5)

In [ ]:
def define_transformer_model(vocab_size, max_length, d_model=1024, num_heads=8, ff_dim=2048, num_transformer_blocks=6):
    class PositionalEncoding(Layer):
        def __init__(self, position, d_model):
            super(PositionalEncoding, self).__init__()
            self.pos_encoding = self.positional_encoding(position, d_model)

        def get_angles(self, position, i, d_model):
            angles = 1 / tf.pow(10000, (2 * (i // 2)) / tf.cast(d_model, tf.float32))
            return position * angles

        def positional_encoding(self, position, d_model):
            angle_rads = self.get_angles(
                position=tf.range(position, dtype=tf.float32)[:, tf.newaxis],
                i=tf.range(d_model, dtype=tf.float32)[tf.newaxis, :],
                d_model=d_model)

            sines = tf.math.sin(angle_rads[:, 0::2])
            cosines = tf.math.cos(angle_rads[:, 1::2])

            pos_encoding = tf.concat([sines, cosines], axis=-1)
            pos_encoding = pos_encoding[tf.newaxis, ...]

            return tf.cast(pos_encoding, tf.float32)

        def call(self, inputs):
            return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

        def get_config(self):
            config = super().get_config()
            config.update({
                "position": self.pos_encoding.shape[1],
                "d_model": self.pos_encoding.shape[2]
            })
            return config

    def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
        x = LayerNormalization(epsilon=1e-6)(inputs)
        x = layers.MultiHeadAttention(
            key_dim=head_size, num_heads=num_heads, dropout=dropout
        )(x, x)
        x = Dropout(dropout)(x)
        res = x + inputs

        x = LayerNormalization(epsilon=1e-6)(res)
        x = Dense(ff_dim, activation="relu")(x)
        x = Dropout(dropout)(x)
        x = Dense(d_model)(x)
        x = Dropout(dropout)(x)
        return x + res

    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(d_model, activation='relu')(fe1)

    fe2_expanded = layers.Reshape((1, d_model))(fe2)

    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, d_model, mask_zero=False)(inputs2)
    se2 = Dropout(0.1)(se1)

    se2 = PositionalEncoding(max_length, d_model)(se2)

    combined_input = layers.Concatenate(axis=1)([fe2_expanded, se2])

    transformer_output = combined_input
    for _ in range(num_transformer_blocks):
        transformer_output = transformer_encoder(
            transformer_output,
            head_size=d_model // num_heads,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout=0.1
        )

    context = transformer_output[:, -1, :]

    decoder = Dense(d_model, activation='relu')(context)
    outputs = Dense(vocab_size, activation='softmax')(decoder)

    model = Model(inputs=[inputs1, inputs2], outputs=outputs)

    model.compile(
        loss=masked_loss,
        optimizer=tf.keras.optimizers.AdamW(learning_rate=0.0005, weight_decay=0.002),
        metrics=[top_5_accuracy]
    )

    print(model.summary())
    return model

model = define_transformer_model(vocab_size, max_length)

In [ ]:
val_filename = os.path.join(dataset_text, "Flickr_8k.devImages.txt")
val_imgs = load_photos(val_filename)
val_descriptions = load_clean_descriptions("descriptions.txt", val_imgs)
val_features = load_features(val_imgs)

In [ ]:
epochs = 30
steps = len(train_descriptions)
val_steps = len(val_descriptions)

train_generator = data_generator(train_descriptions, train_features, tokenizer, max_length)
val_generator = data_generator(val_descriptions, val_features, tokenizer, max_length)

history = model.fit(
    train_generator,
    epochs=epochs,
    steps_per_epoch=steps,
    validation_data=val_generator,
    validation_steps=val_steps,
    verbose=1,
)

In [ ]:
plt.figure(figsize=(14, 5), dpi=300)

plt.subplot(1, 2, 1)
plt.plot(history.history['top_5_accuracy'], label='Train Top 5 Accuracy')
plt.plot(history.history['val_top_5_accuracy'], label='Validation Top 5 Accuracy')
plt.title(f'Top 5 Accuracy per Epoch', size=18, weight='bold')
plt.xlabel('Epoch', size=16, weight='bold')
plt.ylabel('Top 5 Accuracy', size=16, weight='bold')
plt.legend(prop={'weight': 'bold', 'size': 14})
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title(f'Masked Loss per Epoch', size=18, weight='bold')
plt.xlabel('Epoch', size=16, weight='bold')
plt.ylabel('Loss', size=16, weight='bold')
plt.legend(prop={'weight': 'bold', 'size': 14})
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
def extract_features_test(filename, model):
    try:
        image = Image.open(filename)
    except Exception as e:
        print("ERROR: Couldn't open image! Check the image path and extension.")
        return None
    image = image.resize((299, 299))
    image = np.array(image)
    if image.shape[2] == 4:
        image = image[..., :3]
    image = np.expand_dims(image, axis=0)
    image = image / 127.5
    image = image - 1.0
    feature = model.predict(image)
    return feature

def word_for_id(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None

def generate_caption(model, tokenizer, photo, max_length):
    in_text = 'start'

    if photo.ndim == 1:
        photo = np.expand_dims(photo, axis=0)

    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)

        pred = model.predict([photo, sequence], verbose=0)
        pred = np.argmax(pred)
        word = word_for_id(pred, tokenizer)
        if word is None:
            break
        in_text += ' ' + word
        if word == 'end':
            break
    return in_text

In [ ]:
test_img_path = 'Flicker8k_Dataset/111537222_07e56d5a30.jpg'
img = Image.open(test_img_path)
img

In [ ]:
xception_model = Xception(include_top=False, pooling="avg")
photo_feature = extract_features_test(test_img_path, xception_model)
generated_caption = generate_caption(model, tokenizer, photo_feature, max_length)
print("Generated Caption for Test Image:")
print(generated_caption.replace('start', '').replace('end', '').strip())

In [ ]:
test_filename = os.path.join(dataset_text, "Flickr_8k.testImages.txt")
test_imgs = load_photos(test_filename)
test_descriptions = load_clean_descriptions("descriptions.txt", test_imgs)
test_features = load_features(test_imgs)

In [ ]:
def evaluate_model(model, tokenizer, xception_model, test_features, test_descriptions, max_length):

    smoothing = SmoothingFunction().method1
    rouge_evaluator = Rouge()

    bleu1_scores = []
    bleu2_scores = []
    bleu3_scores = []
    bleu4_scores = []

    rouge_scores_list = []

    for img_id, feature in test_features.items():
        y_pred = generate_caption(model, tokenizer, feature, max_length)
        y_pred_clean = y_pred.replace('start', '').replace('end', '').strip()
        if not y_pred_clean.strip():
            print(f"Skipping ROUGE evaluation for {img_id} due to empty prediction.")
            continue

        pred_tokens = y_pred_clean.split()

        references = test_descriptions[img_id]
        ref_tokens_list = [ref.replace('start', '').replace('end', '').strip().split() for ref in references]

        bleu1 = sentence_bleu(ref_tokens_list, pred_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothing)
        bleu2 = sentence_bleu(ref_tokens_list, pred_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
        bleu3 = sentence_bleu(ref_tokens_list, pred_tokens, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smoothing)
        bleu4 = sentence_bleu(ref_tokens_list, pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)

        bleu1_scores.append(bleu1)
        bleu2_scores.append(bleu2)
        bleu3_scores.append(bleu3)
        bleu4_scores.append(bleu4)

        rouge_scores_img = []
        for ref in references:
            ref_clean = ref.replace('start', '').replace('end', '').strip()
            scores = rouge_evaluator.get_scores(y_pred_clean, ref_clean)[0]
            rouge_scores_img.append(scores)

        rouge_avg = {}
        for key in rouge_scores_img[0].keys():
            rouge_avg[key] = np.mean([score[key]['f'] for score in rouge_scores_img])
        rouge_scores_list.append(rouge_avg)

        print(f"Image ID: {img_id}")
        print("Generated Caption:", y_pred_clean)
        print("BLEU-1:", bleu1, "BLEU-2:", bleu2, "BLEU-3:", bleu3, "BLEU-4:", bleu4)
        print("ROUGE-1:", round(rouge_avg.get("rouge-1", 0), 4),
              "ROUGE-2:", round(rouge_avg.get("rouge-2", 0), 4),
              "ROUGE-L:", round(rouge_avg.get("rouge-l", 0), 4))
        print("-" * 50)

    avg_bleu1 = np.mean(bleu1_scores)
    avg_bleu2 = np.mean(bleu2_scores)
    avg_bleu3 = np.mean(bleu3_scores)
    avg_bleu4 = np.mean(bleu4_scores)

    avg_rouge = {}
    for key in rouge_scores_list[0].keys():
        avg_rouge[key] = np.mean([r[key] for r in rouge_scores_list])

    print("\nAverage BLEU-1 Score:", avg_bleu1)
    print("Average BLEU-2 Score:", avg_bleu2)
    print("Average BLEU-3 Score:", avg_bleu3)
    print("Average BLEU-4 Score:", avg_bleu4)
    print("Average ROUGE-1:", round(avg_rouge.get("rouge-1", 0), 4),
          "Average ROUGE-2:", round(avg_rouge.get("rouge-2", 0), 4),
          "Average ROUGE-L:", round(avg_rouge.get("rouge-l", 0), 4))

    results = {
        'BLEU': {
            'BLEU-1': avg_bleu1,
            'BLEU-2': avg_bleu2,
            'BLEU-3': avg_bleu3,
            'BLEU-4': avg_bleu4,
        },
        'ROUGE': {
            'ROUGE-1': round(avg_rouge.get("rouge-1", 0), 4),
            'ROUGE-2': round(avg_rouge.get("rouge-2", 0), 4),
            'ROUGE-L': round(avg_rouge.get("rouge-l", 0), 4)
        }
    }
    return results

In [ ]:
results = evaluate_model(model, tokenizer, xception_model, test_features, test_descriptions, max_length)
print("Evaluation Results :")
print(results)

In [ ]:
metrics = []
scores = []

for metric, score in results['BLEU'].items():
    metrics.append(metric)
    scores.append(score)

for metric, score in results['ROUGE'].items():
    metrics.append(metric)
    scores.append(score)

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, scores)

plt.xlabel('Evaluation Metrics')
plt.ylabel('Score')
plt.title('Evaluation Metrics for Captioning Model')
plt.ylim([0, 1])

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.02, f'{height:.4f}', ha='center', fontweight='bold')

plt.show()

In [ ]:
def evaluate_cider(model, tokenizer, xception_model, test_features, test_descriptions, max_length):
    cider_scorer = Cider()

    candidate_dict = {}
    reference_dict = {}

    for img_id, feature in test_features.items():
        y_pred = generate_caption(model, tokenizer, feature, max_length)
        y_pred_clean = y_pred.replace('start', '').replace('end', '').strip()

        candidate_dict[img_id] = [y_pred_clean]

        references = [ref.replace('start', '').replace('end', '').strip() for ref in test_descriptions[img_id]]
        reference_dict[img_id] = references

    cid_score, cid_scores = cider_scorer.compute_score(reference_dict, candidate_dict)
    print("CIDEr Score: {:.4f}".format(cid_score))

    return cid_score, cid_scores

cid_score, cid_scores = evaluate_cider(model, tokenizer, xception_model, test_features, test_descriptions, max_length)

print(cid_score)
print(cid_scores)

In [ ]:
if isinstance(cid_scores, dict):
    scores_list = list(cid_scores.values())
else:
    scores_list = cid_scores

plt.figure(figsize=(10, 6))
plt.hist(scores_list, bins=20, color='skyblue', edgecolor='black')
plt.xlabel("CIDEr Score")
plt.ylabel("Number of Images")
plt.title("Distribution of CIDEr Scores")
plt.show()

plt.figure(figsize=(8, 6))
sns.boxplot(x=scores_list, color='lightgreen')
plt.xlabel("CIDEr Score")
plt.title("Boxplot of CIDEr Scores")
plt.show()

In [ ]:
def evaluate_spice(model, tokenizer, test_features, test_descriptions, max_length):
    spice_scorer = Spice()

    candidate_dict = {}
    reference_dict = {}

    for img_id, feature in test_features.items():
        candidate = generate_caption(model, tokenizer, feature, max_length)
        candidate_clean = candidate.replace('start', '').replace('end', '').strip()
        candidate_dict[img_id] = [candidate_clean]

        references = [ref.replace('start', '').replace('end', '').strip() for ref in test_descriptions[img_id]]
        reference_dict[img_id] = references

        print(f"Image ID: {img_id}")
        print("Candidate:", candidate_clean)
        print("References:", references)
        print("-" * 50)

    spice_score, spice_scores = spice_scorer.compute_score(reference_dict, candidate_dict)
    print("Overall SPICE Score: {:.4f}".format(spice_score))
    return spice_score, spice_scores

spice_score, spice_scores = evaluate_spice(model, tokenizer, test_features, test_descriptions, max_length)

In [ ]:
print("SPICE Scores:", spice_scores)
print("Number of SPICE Scores:", len(spice_scores))